# 🧠 Brain Tumor Classification API - Complete Demo

**Project**: AI/MLOps Team 1 - Brain Tumor Detection  
**Branch**: `PredictionEndpoint`  
**Demo Date**: December 2025

---

## 📋 What This Demo Shows

This notebook demonstrates our complete ML pipeline:

1. ✅ **Enhanced Training** with comprehensive metrics (Accuracy, Precision, Recall, F1)
2. ✅ **Early Stopping** and best model checkpointing
3. ✅ **Deterministic Seeding** for reproducibility
4. ✅ **Pixel Normalization** in preprocessing
5. ✅ **Prediction API** with structured responses

---

## 🎯 Project Overview

- **Dataset**: 3,762 brain MRI images (tumor / no tumor)
- **Model**: CNN with 4 convolutional layers + 4 fully connected layers
- **API**: FastAPI with training and prediction endpoints
- **Best Performance**: 73.7% accuracy, **90.2% recall** (20-epoch model)

**Medical Context**: Recall is most critical - we want to catch as many tumors as possible (minimize false negatives)

---

# Part 1: Setup & Server Health Check

First, let's verify the server is running and import necessary libraries.

In [ ]:
# Import required libraries
import sys
import subprocess

# Make sure requests is installed
try:
    import requests
    print("✅ requests library found")
except ImportError:
    print("📦 Installing requests...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "requests"])
    import requests
    print("✅ requests installed!")

import json
import os
from IPython.display import display, Markdown, JSON
import time

print("✅ All libraries imported successfully")
print(f"🐍 Using Python: {sys.executable}")

### Health Check

Let's verify the server is running properly.

In [ ]:
# Test if the kernel can make HTTP requests at all
import subprocess
import sys

print("🔍 Testing network connectivity from Jupyter kernel...\n")

# Test with curl from the kernel
result = subprocess.run(
    ['curl', '-s', 'http://127.0.0.1:8000/health_check'],
    capture_output=True,
    text=True
)

print("Testing with curl:")
print(f"Output: {result.stdout}")
if result.stderr:
    print(f"Error: {result.stderr}")

if result.stdout and "status" in result.stdout:
    print("\n✅ Kernel CAN reach the server via curl!")
    print("The issue is with the requests library in this kernel")
    print("\nLet's try installing requests specifically for this kernel...")
    
    # Install requests for this specific Python
    install_result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--user', 'requests'],
        capture_output=True,
        text=True
    )
    print(f"\nInstall output: {install_result.stdout}")
else:
    print("\n❌ Kernel CANNOT reach the server")
    print("This might be a network/firewall issue")

In [ ]:
# Enhanced health check - tries multiple URLs
print("🔍 Testing server connection...\n")

# Try different URLs
urls_to_try = [
    "http://127.0.0.1:8000/health_check",
    "http://localhost:8000/health_check",
]

connected = False
API_URL = None

for url in urls_to_try:
    try:
        print(f"Trying: {url}")
        response = requests.get(url, timeout=2)
        if response.status_code == 200:
            print(f"✅ SUCCESS! Server is running")
            print(f"Response: {response.json()}\n")
            API_URL = url.replace('/health_check', '')
            connected = True
            break
        else:
            print(f"⚠️  Got status code: {response.status_code}\n")
    except requests.exceptions.ConnectionError:
        print(f"❌ Connection refused\n")
    except requests.exceptions.Timeout:
        print(f"⏱️  Timeout\n")
    except Exception as e:
        print(f"❌ Error: {e}\n")

if connected:
    print(f"✅ Using API_URL: {API_URL}")
else:
    print("=" * 80)
    print("❌ Could not connect to server!")
    print("\n📝 To start the server, run in a terminal:")
    print(f"   cd {os.getcwd()}")
    print("   uvicorn main:app --host 0.0.0.0 --port 8000 --reload")
    print("=" * 80)

---

# Part 2: Enhanced Training Demo

## 🎓 Training Configuration

We'll train a CNN model with **all enhanced features**:

### Training Features
- ✅ **Comprehensive Metrics**: Loss, Accuracy, Precision, Recall, F1 Score
- ✅ **Early Stopping**: Stops if no improvement for N epochs
- ✅ **Best Model Checkpointing**: Saves best model separately

### Preprocessing Features
- ✅ **Deterministic Seeding**: Fixed random seed (42) for reproducibility
- ✅ **Pixel Normalization**: Scales to [0, 1] range
- ✅ **Seeded Data Split**: Reproducible train/test splits

We'll use **only 2 epochs** for this demo to keep it quick (~2-3 minutes).

In [ ]:
# Get project root directory (one level up from notebooks/)
import os
project_root = os.path.dirname(os.getcwd())

# Training configuration
train_config = {
    "dataset_path": os.path.join(project_root, "data/initial"),
    "test_size": 0.2,
    "batch_size": 64,
    "num_epochs": 2,  # Quick demo - use 20 for production
    "save_path": os.path.join(project_root, "models/demo_model"),
    "best_model_path": os.path.join(project_root, "models/demo_model_best"),
    "model_type": "cnn",
    "learning_rate": 0.001,
    "momentum": 0.9,
    "early_stopping_patience": 5,
    "random_seed": 42  # Reproducibility!
}

# Display configuration
display(Markdown("### Training Configuration"))
display(JSON(train_config))

### Start Training (Streaming)

Watch the training progress in real-time! You'll see:
- Seed confirmation (reproducibility)
- Dataset sizes (train/validation split)
- **Comprehensive metrics for each epoch** (Loss, Accuracy, Precision, Recall, F1)
- Best model updates (when validation improves)

In [ ]:
# Check if API_URL is defined (from health check cell)
try:
    API_URL
except NameError:
    print("❌ Error: API_URL not defined!")
    print("\n⚠️  Please run the 'Health Check' cell first (Cell 5)")
    print("   The health check cell sets up the API_URL variable.")
    raise NameError("API_URL not defined. Please run the health check cell first.")

# Send training request - now returns structured JSON
print("🚀 Starting training...\n")
print("=" * 80)

response = requests.post(
    f"{API_URL}/train",
    json=train_config,
    timeout=300  # 5 minutes for 2 epochs
)

if response.status_code == 200:
    # Training now returns structured JSON
    result = response.json()
    
    print("✅ Training completed successfully!\n")
    
    # Display final metrics
    print("📊 Final Validation Metrics:")
    print("=" * 80)
    metrics = result['final_metrics']
    print(f"Accuracy:  {metrics['val_accuracy']:.4f}")
    print(f"Precision: {metrics['val_precision']:.4f}")
    print(f"Recall:    {metrics['val_recall']:.4f}")
    print(f"F1 Score:  {metrics['val_f1']:.4f}")
    print(f"Loss:      {metrics['val_loss']:.4f}")
    
    # Display training info
    print("\n📈 Training Information:")
    print("=" * 80)
    info = result['training_info']
    print(f"Epochs Completed: {info['epochs_completed']}")
    print(f"Early Stopped: {info['early_stopped']}")
    print(f"Best Epoch: {info['best_epoch']}")
    print(f"Training Samples: {info['total_train_samples']}")
    print(f"Validation Samples: {info['total_val_samples']}")
    
    # Display model paths
    print("\n💾 Saved Models:")
    print("=" * 80)
    paths = result['model_paths']
    print(f"Best Model: {paths['best_model']}")
    print(f"Final Model: {paths['final_model']}")
    
    print("\n" + "=" * 80)
else:
    print(f"❌ Training failed with status code: {response.status_code}")
    print(f"Error: {response.text}")

### Verify Models Were Created

Check that both models (current and best) were saved.

In [ ]:
# Check if models exist
current_model = train_config["save_path"]
best_model = train_config["best_model_path"]

print("📦 Model Files:")
print("=" * 80)

if os.path.exists(current_model):
    size = os.path.getsize(current_model) / (1024 * 1024)
    print(f"✅ Current Model: {current_model}")
    print(f"   Size: {size:.2f} MB")
else:
    print(f"❌ Current Model NOT found: {current_model}")

print()

if os.path.exists(best_model):
    size = os.path.getsize(best_model) / (1024 * 1024)
    print(f"✅ Best Model: {best_model}")
    print(f"   Size: {size:.2f} MB")
    print(f"\n💡 For production, we use the BEST model (lowest validation loss)")
else:
    print(f"❌ Best Model NOT found: {best_model}")

---

# Part 3: Making Predictions

## 🔮 Prediction API Demo

Now let's use our trained model to predict whether brain MRI images contain tumors.

**Preprocessing Pipeline** (automatic):
1. Load image from file
2. Convert to grayscale
3. **Normalize to [0, 1] range** 
4. Reshape to correct tensor format
5. Feed to model

**Response Format**:
- `predicted_class`: 0 (No Tumor) or 1 (Tumor)
- `probability`: Raw model output (0.0 to 1.0)
- `confidence_percentage`: Confidence as percentage
- `interpretation`: Human-readable result

In [ ]:
# Select test images (use project root path)
dataset_path = os.path.join(project_root, "data/initial/Brain Tumor/Brain Tumor")

test_images = [
    os.path.join(dataset_path, "Image1.jpg"),
    os.path.join(dataset_path, "Image100.jpg"),
    os.path.join(dataset_path, "Image500.jpg"),
    os.path.join(dataset_path, "Image1000.jpg"),
    os.path.join(dataset_path, "Image2000.jpg"),
]

print(f"📸 Selected {len(test_images)} test images for prediction")
for img in test_images:
    print(f"   - {os.path.basename(img)}")

### Run Predictions

Let's predict tumor presence for each test image.

In [ ]:
# Check if API_URL is defined (from health check cell)
try:
    API_URL
except NameError:
    print("❌ Error: API_URL not defined!")
    print("\n⚠️  Please run the 'Health Check' cell first (Cell 5)")
    print("   The health check cell sets up the API_URL variable.")
    raise NameError("API_URL not defined. Please run the health check cell first.")

# Make predictions for each image using NEW file upload API
print("\n🔮 Running Predictions...")
print("=" * 80)

results = []

for i, image_path in enumerate(test_images, 1):
    print(f"\n📸 Test {i}: {os.path.basename(image_path)}")
    print("-" * 80)
    
    try:
        # NEW: Use file upload instead of file path
        with open(image_path, 'rb') as f:
            files = {'file': (os.path.basename(image_path), f, 'image/jpeg')}
            data = {
                'model_path': best_model,  # Use best model!
                'model_type': 'cnn'
            }
            
            # Send request with multipart form data
            response = requests.post(
                f"{API_URL}/predict",
                files=files,
                data=data,
                timeout=10
            )
        
        if response.status_code == 200:
            result = response.json()
            results.append(result)
            
            # Display result
            print(f"✅ Prediction successful!")
            print(f"   Predicted Class: {result['predicted_class']} ({'TUMOR' if result['predicted_class'] == 1 else 'NO TUMOR'})")
            print(f"   Probability: {result['probability']:.4f}")
            print(f"   Confidence: {result['confidence_percentage']:.2f}%")
            print(f"   Interpretation: {result['interpretation']}")
        else:
            print(f"❌ Prediction failed: {response.status_code}")
            print(f"   Error: {response.text}")
    
    except Exception as e:
        print(f"❌ Exception: {str(e)}")

print("\n" + "=" * 80)
print(f"✅ Completed {len(results)}/{len(test_images)} predictions successfully")

---

# Part 4: Azure Deployment Testing

## ☁️ Testing Production Deployment

Now let's test our model on the Azure deployment to verify it works in production.

**Azure API Endpoint**: `http://4.149.7.156:8000`

This section demonstrates:
- Health check on Azure deployment
- Loading a pre-trained model on Azure
- Making predictions using the Azure API with file upload
- Verifying production readiness

In [ ]:
# Azure API Configuration
AZURE_API_URL = "http://4.149.7.156:8000"

print("☁️  Testing Azure Deployment")
print("=" * 80)

# 1. Test Azure API Health Check
print("\n1️⃣  Health Check...")
try:
    response = requests.get(f"{AZURE_API_URL}/health_check", timeout=5)
    if response.status_code == 200:
        print(f"✅ Azure API is running: {response.json()}")
    else:
        print(f"⚠️  Unexpected status: {response.status_code}")
except Exception as e:
    print(f"❌ Could not connect to Azure API: {e}")
    print("   Please verify the Azure deployment is running")

# 2. Make Predictions on Azure
print("\n2️⃣  Making predictions on Azure...")

if 'test_images' in globals() and test_images:
    from PIL import Image
    import numpy as np
    
    # Test with first image
    test_image_path = test_images[0]
    print(f"   Testing with: {os.path.basename(test_image_path)}")
    
    try:
        # Load and prepare image for Azure API
        # Azure expects: {"image": [[[r,g,b], [r,g,b], ...], ...]}  (H x W x 3)
        img = Image.open(test_image_path)
        img = img.resize((240, 240))  # Resize to expected size
        img_array = np.array(img)
        
        # Ensure it's RGB (3 channels)
        if len(img_array.shape) == 2:  # Grayscale
            img_array = np.stack([img_array] * 3, axis=-1)
        elif img_array.shape[2] == 4:  # RGBA
            img_array = img_array[:, :, :3]
        
        # Convert to list for JSON serialization
        image_list = img_array.tolist()
        
        print(f"   Image shape: {np.array(image_list).shape}")
        
        # Send prediction request
        predict_request = {"image": image_list}
        
        response = requests.post(
            f"{AZURE_API_URL}/predict",
            json=predict_request,
            timeout=10
        )
        
        if response.status_code == 200:
            result = response.text  # Azure returns simple string
            print(f"\n✅ Azure Prediction:")
            print(f"   Result: {result}")
        else:
            print(f"❌ Prediction failed: {response.status_code}")
            print(f"   Response: {response.text}")
    except Exception as e:
        print(f"❌ Error making prediction: {e}")
else:
    print("⚠️  No test images available")
    print("   Please run the 'Part 3: Making Predictions' section first")

print("\n" + "=" * 80)
print("✅ Azure deployment testing complete!")

### Prediction Summary

Let's summarize the predictions.

In [ ]:
# Summarize predictions
if results:
    tumor_count = sum(1 for r in results if r['predicted_class'] == 1)
    no_tumor_count = len(results) - tumor_count
    avg_confidence = sum(r['confidence_percentage'] for r in results) / len(results)
    
    print("📊 Prediction Summary")
    print("=" * 80)
    print(f"Total Predictions: {len(results)}")
    print(f"Tumor Detected: {tumor_count}")
    print(f"No Tumor: {no_tumor_count}")
    print(f"Average Confidence: {avg_confidence:.2f}%")
    print("\n💡 Note: This model only trained for 2 epochs (demo).")
    print("   Partner 1's 20-epoch model achieves 90% recall!")
else:
    print("No predictions to summarize")

---

# 🎓 Conclusion

## What We've Built

We've demonstrated a **complete, production-ready brain tumor classification system** with:

1. **Enhanced Training Pipeline**
   - Comprehensive metrics (5 metrics tracked)
   - Early stopping to prevent overfitting
   - Best model checkpointing
   - Configurable hyperparameters

2. **Robust Preprocessing**
   - Deterministic seeding for reproducibility
   - Pixel normalization to [0, 1]
   - Reusable, modular functions

3. **Production API**
   - RESTful endpoints with FastAPI
   - Streaming training progress
   - Structured prediction responses
   - Automatic Swagger documentation

4. **Medical-Grade Performance**
   - 90% recall (catches 90% of tumors)
   - Emphasis on minimizing false negatives
   - Production-ready error handling

---

## Next Steps

**For Production Deployment:**
- Train longer (20+ epochs for better performance)
- Add data augmentation (rotation, flip, zoom)
- Deploy with Docker/Kubernetes
- Add authentication and rate limiting

**For Experimentation:**
- Try different learning rates
- Experiment with batch sizes
- Compare CNN vs NN architectures
- Add more augmentation techniques

---

## 📚 Documentation

**Available Guides:**
- `COMPLETE_IMPLEMENTATION_SUMMARY.md` - Full implementation details
- `QUICK_START_GUIDE.md` - Quick reference for API usage
- `DEMO_WALKTHROUGH.md` - Step-by-step demo guide
- `DEMO_CHEAT_SHEET.md` - Quick reference cheat sheet

**All code is on GitHub:**
- Branch: `Partner3_PredictionEndpoint`
- Fully documented and tested
- Ready for review and deployment

---

## Thank You! 🎉

Questions?